# 1 Dataset Introduction

Financial markets are closely linked: the price of one asset is often affected by movements in stock markets, other commodities, currencies and the wider economy. Gold is a common example. Investors often treat it as a "safe haven" and buy it when stock markets fall or inflation rises, and because it is priced in US dollars it also responds to changes in the dollar's value and in interest rates.

This dataset brings these factors together in one daily record, covering two major US stock indices (the S&P 500 and NASDAQ), five commodities (gold, silver, oil, platinum and palladium), two exchange rates and three US economic indicators. This makes it suitable for studying how the assets relate to one another and for building models that predict a future price, such as gold, from the behaviour of the rest of the market.

## 1.1 Dataset Identification

The dataset was retrieved from Kaggle, where it was published by the user franciscogcc under the title "Gold Price Regression". It is a daily time series of financial information for market indices, commodities, economic indicators and foreign exchange rates. The market indices and commodities are represented by the price of the exchange-traded fund (ETF) that tracks each one, rather than by the index or spot price itself. Kaggle does not verify uploaded datasets, so this source should be treated with more caution than a curated repository such as UC Irvine. The dataset can be found at the following URL: 

https://www.kaggle.com/datasets/franciscogcc/financial-data

The dataset contains forty-seven (47) attributes over 3,904 rows, covering trading days from 14 January 2010 to 23 October 2024. The file does not define a target. As the dataset was published for regression on the gold price, gold close is treated as the target, with the remaining forty-six (46) attributes as features. The attributes are defined as the following types:

- <mark>Date</mark>: date Presented as a string in the format YYYY-MM-DD. For example: 2010-01-14.

- <mark>Numerical (Continuous)</mark>: the open, high, low and close prices of sp500, nasdaq, silver, oil, platinum, palladium and gold; the daily range (high-low) of each asset except gold; the exchange rates usd_chf and eur_usd; and the macroeconomic indicators us_rates_%, CPI and GDP (39 attributes in total). Prices and rates can take any real value and have no defined upper limit.

- <mark>Numerical (Discrete)</mark>: sp500 volume, nasdaq volume, silver volume, oil volume, platinum volume, palladium volume, gold volume. Volumes are counts of units traded, so they only take whole-number values, although the file stores them as decimals.

There are no categorical attributes. Around 5% of rows are empty apart from the date; these appear to be non-trading days. us_rates_%, CPI and GDP are published monthly or quarterly, so they are present on only around 5%, 5% and 1% of rows respectively. They would need to be forward-filled or removed before modelling. Table 1 summarises the attributes.

| Attribute(s) | Data type | Variable type | Description |
|---|---|---|---|
| date | String | Date | Trading day (YYYY-MM-DD) |
| [asset] open / high / low / close | Float | Numerical (continuous) | ETF price at open, highest, lowest and at close of the day |
| [asset] high-low | Float | Numerical (continuous) | Daily price range (high minus low) |
| [asset] volume | Float | Numerical (discrete) | Number of units traded that day |
| usd_chf, eur_usd | Float | Numerical (continuous) | Exchange rates |
| us_rates_% | Float | Numerical (continuous) | US interest rate (%) |
| CPI, GDP | Float | Numerical (continuous) | US Consumer Price Index and Gross Domestic Product |
| gold close | Float | Numerical (continuous) | Target: gold ETF closing price |

## 1.2. Supervised Learning Task Identification

**Financial Regression**

Herein, the objective of the supervised learning task is to use regression techniques to accurately predict the closing price of gold (gold close). This can also be determined from the following:

- The target variable, gold close, is numerical (continuous), meaning it can take any value within a range rather than belonging to a fixed set of categories.
- The dataset contains no categorical attributes that could act as a class label; all attributes other than date are numerical.
- The data is an ordered daily time series, so the task is to estimate a future value from earlier values of gold and the other assets, rates and indicators.

A classification task can also be performed by deriving a categorical target from the data, such as whether the price of gold rises or falls on the following day. However, this is not the dataset's primary purpose.

# 2. Exploratory Data Analysis

With the identification of the dataset, and the supervised learning task being defined as Financial Regression and regression respectively, exploratory data analysis (EDA) can be undertaken to facilitate a deeper understanding of the structure of the data, as well as allowing for identification of issues with data quality (IBM, 2024). This is particularly important for this dataset, as it contains missing values on non-trading days and macroeconomic indicators (us_rates_%, CPI and GDP) that are only recorded monthly or quarterly. Furthermore, by creating visualisations of the dataset, it is easier to identify any patterns, trends, or correlations that are present between gold and the other assets, rates and indicators. With conclusions drawn from this exploration, suitable machine learning algorithms and models can be identified, and features can be engineered to improve the performance of the chosen model(s). These model(s) can then be evaluated for their efficacy when provided data of a similar format to the original dataset.

Within the following section, questions and assumptions about the dataset will be proposed. Then an explanation of how data will be split into training, validation, and testing sets will be provided; as the data is a time series, this split will be made chronologically rather than at random, so that the model is never trained on data from after the period it is tested on. This will be followed by EDA, defining the process and results, which will then be measured against the previously proposed questions and assumptions.

### 2.1 Question(s) identification

Research questions should allow the researcher or investigator to set out what they want to know as a hypothesis, regardless of whether the result support the hypothesis or not (Mayo et al., 2013). By providing well posed questions and assumptions the EDA process will be able to support or reject their claims (Lane, 2018). Table 2 below provides several questions related to the dataset and their associated assumptions.

| ID | Question | Assumption |
|---|---|---|
| RQ1 | Does the price of gold move with the stock market (sp500 and nasdaq)? | Gold is often bought as a "safe haven" when stock markets fall, so it is expected that daily changes in gold will show little or a slightly negative relationship with daily changes in the stock indices. Over the full period, however, both may show a positive relationship simply because both rose over time. |
| RQ2 | Do other precious metals (silver, platinum and palladium) move with gold? | It is expected that silver will show a strong positive correlation with gold, as both are traded as precious metals and stores of value. Platinum and palladium are expected to show a weaker relationship, as their prices depend more on industrial demand such as vehicle manufacturing. |
| RQ3 | Do interest rates (us_rates_%) impact the price of gold? | Gold pays no interest, so it is expected that higher interest rates will make it less attractive than interest-bearing assets, resulting in a negative relationship between interest rates and the price of gold. |
| RQ4 | Does the strength of the US dollar (eur_usd and usd_chf) impact the price of gold? | As gold is priced in US dollars, it is expected that a weaker dollar will coincide with a higher gold price. This would appear as a positive relationship with eur_usd and a negative relationship with usd_chf. |
| RQ5 | Does inflation (CPI) impact the price of gold? | Gold is commonly treated as a protection against inflation, so it is expected that periods of rising CPI will coincide with a rising gold price. As CPI is only recorded monthly, this relationship is expected to be visible over long periods rather than day to day. |
| RQ6 | Does the previous day's price of gold predict the next day's price? | It is expected that the previous day's closing price will be the strongest single predictor of the next day's price, as daily price changes are usually small relative to the price itself. |

2.2 Splitting the dataset
The dataset will be split into three groups; the training set, the validation set, and the testinG set. It is important to train the model on a distinct set of data to the testing data, otherwise the model may memorise the training data and overfit to it, making it unreliable in real-world scenarios where it is expected to perform on unseen data. (Apicella et al., 2024). This is also referred to as data-leakage, which refers to the unintentional provision of data from outside the training dataset during model training.

To split the data, the module train_test_split from Scikit learn is used to allocate two subsets of data. This initial split creates a training and testing dataset, the code for which is shown in Figure 2; to allow for accurate replication of this split, the random_state parameter is used. This parameter is used to control the random number generator for the random state. In this case, the number 42 has been used, and this value can be used again to perform the same split (Scikit Learn, 2024.)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import accuracy_score, recall_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
DATA_PATH = "../datasets/"
price = pd.read_csv(DATA_PATH + "financial_regression.csv")
print(price.shape)
price.head

(3904, 47)


<bound method NDFrame.head of             date  sp500 open  sp500 high  sp500 low  sp500 close  \
0     2010-01-14      114.49    115.1400   114.4200       114.93   
1     2010-01-15      114.73    114.8400   113.2000       113.64   
2     2010-01-18         NaN         NaN        NaN          NaN   
3     2010-01-19      113.62    115.1300   113.5900       115.06   
4     2010-01-20      114.28    114.4500   112.9800       113.89   
...          ...         ...         ...        ...          ...   
3899  2024-10-17      585.91    586.1200   582.1600       582.35   
3900  2024-10-18      584.07    585.3900   582.5800       584.59   
3901  2024-10-21      583.85    584.8500   580.6001       583.63   
3902  2024-10-22      581.05    584.5000   580.3800       583.32   
3903  2024-10-23      581.26    581.7086   574.4150       577.99   

      sp500 volume  sp500 high-low  nasdaq open  nasdaq high  nasdaq low  ...  \
0      115646960.0          0.7200        46.26       46.520     46.2200